In [18]:
import pandas as pd
import numpy as np

df = pd.read_csv("cardio_train.csv", sep=';')
df.head()

df['age_years'] = (df['age'] / 365.25).astype(int)

df.columns = df.columns.str.strip()

print(df[['age', 'age_years']].head())



     age  age_years
0  18393         50
1  20228         55
2  18857         51
3  17623         48
4  17474         47


In [19]:
df_clean = df[(df['height'] >= 140) & (df['height'] <= 200)].copy()
df_clean = df_clean[(df_clean['weight'] >= 40) & (df_clean['weight'] <= 180)]

In [20]:
df_clean = df_clean[(df_clean['ap_hi'] >= 80) & (df_clean['ap_hi'] <= 220)]
df_clean = df_clean[(df_clean['ap_lo'] >= 50) & (df_clean['ap_lo'] <= 130)]
df_clean[df_clean['ap_hi'] > df_clean['ap_lo']]


,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0,50
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1,55
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1,51
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1,48
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0,47
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69995,99993,19240,2,168,76.0,120,80,1,1,1,0,1,0,52
69996,99995,22601,1,158,126.0,140,90,2,2,0,0,1,1,61
69997,99996,19066,2,183,105.0,180,90,3,1,0,1,0,1,52
69998,99998,22431,1,163,72.0,135,80,1,2,0,0,0,1,61


In [21]:
df_clean['bmi'] = (df_clean['weight'] / ((df_clean['height'] / 100) ** 2)).round(1)

In [22]:
df_clean['map'] = (df_clean['ap_lo'] + ((df_clean['ap_hi'] - df_clean['ap_lo']) / 3)).round(1)

In [23]:
bins = [30, 40, 50, 60, 100]
labels = ['30-39', '40-49', '50-59', '60+']
df_clean['age_group'] = pd.cut(df_clean['age_years'], bins=bins, labels=labels, right=False)

In [24]:
def calc_risk(row):
    score = 0
    if row['cholesterol'] > 1: score += 1
    if row['gluc'] > 1: score += 1
    if row['map'] >= 100: score += 1
    if row['bmi'] >= 30: score += 1
    return score

df_clean['risk_score'] = df_clean.apply(calc_risk, axis=1)

In [25]:
df_clean['risk_profile'] = np.where(df_clean['risk_score'] >= 2, 'High Risk Profile', 'Controlled Profile')

In [26]:
df_clean['cholesterol_label'] = df_clean['cholesterol'].map({1: 'Normal', 2: 'Above Normal', 3: 'Well Above Normal'})
df_clean['gluc_label'] = df_clean['gluc'].map({1: 'Normal', 2: 'Above Normal', 3: 'Well Above Normal'})
df_clean['cardio_label'] = df_clean['cardio'].map({0: 'Healthy', 1: 'Cardiovascular Disease'})

In [27]:
print(f"Filas originales: {len(df)} | Filas tras limpieza: {len(df_clean)}")
print(f"Registros descartados (outliers): {len(df) - len(df_clean)}")

Filas originales: 70000 | Filas tras limpieza: 68480
Registros descartados (outliers): 1520


In [28]:
df_clean.to_csv('cardio_cleaned.csv', index=False)
print(" 'cardio_cleaned.csv' exportado")

 'cardio_cleaned.csv' exportado
